# Model3_Hybrid_Multimodal.ipynb — Group 15 | 7PAM2033
# Model   : Hybrid Multimodal (CNN Image Branch + XGBoost Clinical Branch)
# Purpose : Combine MRI image predictions with clinical data predictions
#           using feature-level fusion. The two datasets have different patients
#           so we train each branch separately then combine via probability
#           averaging at inference time.
#
# Architecture:
#   Branch 1 → EfficientNetB4 on MRI images → class probabilities
#   Branch 2 → XGBoost on clinical CSV data → class probabilities
#   Fusion   → Weighted average of both probability outputs

In [2]:
# ================================================================================
# CELL 1 — Import libraries
# ================================================================================

import os
import random
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13})

logger.info("Libraries loaded.")

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
# ================================================================================
# CELL 2 — Paths and settings
# ================================================================================

MRI_DIR      = Path("../Data/MRI_Augmented")
CLINICAL_CSV = Path("../Data/Clinical_Data.csv")
RESULTS_DIR  = Path("../results")
MODELS_DIR   = Path("../results/models")
PLOTS_DIR    = Path("../results/plots")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# MRI settings — must match what was used in Model 1 and Model 2
MRI_CLASSES = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
N_CLASSES   = len(MRI_CLASSES)
IMG_SIZE    = (224, 224)
IMG_SHAPE   = (224, 224, 3)
BATCH_SIZE  = 32

# Columns to drop from clinical data — not useful for prediction
DROP_COLS        = ["PatientID", "DoctorInCharge"]
TARGET_COL       = "Diagnosis"

# Demographic columns saved separately for fairness evaluation later
SENSITIVE_COLS   = ["Age", "Gender", "Ethnicity", "EducationLevel"]

# Fusion weight — how much to trust each branch
# 0.5/0.5 means equal trust in MRI and clinical predictions
MRI_WEIGHT      = 0.5
CLINICAL_WEIGHT = 0.5

logger.info("Paths configured.")

NameError: name 'Path' is not defined

In [ ]:
# ================================================================================
# CELL 3 — BRANCH 1: Load the best MRI model from Model 2
# ================================================================================
# We reuse the EfficientNetB4 model trained in Model 2 as the MRI branch.
# This avoids retraining and ensures a fair comparison since the same MRI
# model is used in both Model 2 and Model 3.

mri_model_path = MODELS_DIR / "model2_efficientnet_best.h5"

if not mri_model_path.exists():
    # Fall back to Model 1 if Model 2 hasn't been run yet
    mri_model_path = MODELS_DIR / "model1_baseline_cnn_best.h5"
    logger.warning("Model 2 not found — using Model 1 as MRI branch.")

mri_model = tf.keras.models.load_model(str(mri_model_path))
logger.info("MRI model loaded from: %s", mri_model_path)
print(f"MRI branch loaded: {mri_model_path.name}")


In [ ]:
# ================================================================================
# CELL 4 — Get MRI branch predictions
# ================================================================================
# Run the MRI model on the full augmented dataset to get class probabilities.
# These probabilities become the MRI branch input to the fusion layer.

eval_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

mri_gen = eval_datagen.flow_from_directory(
    str(MRI_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=MRI_CLASSES,
    shuffle=False
)

print("Getting MRI predictions...")
mri_gen.reset()
mri_probs  = mri_model.predict(mri_gen, verbose=1)  # shape: (n_samples, 4)
mri_preds  = np.argmax(mri_probs, axis=1)
mri_true   = mri_gen.classes

mri_acc = accuracy_score(mri_true, mri_preds)
print(f"\nMRI branch accuracy: {mri_acc:.4f}")
print(f"MRI predictions shape: {mri_probs.shape}")

In [ ]:
# ================================================================================
# CELL 5 — BRANCH 2: Load and preprocess clinical data
# ================================================================================
# Load the El Kharoua clinical dataset and prepare it for XGBoost training.
# The clinical dataset has different patients from the MRI dataset — that is
# expected. Each branch learns from its own dataset independently.

if not CLINICAL_CSV.exists():
    raise FileNotFoundError(
        f"Clinical CSV not found: {CLINICAL_CSV.resolve()}\n"
        "Please update CLINICAL_CSV path."
    )

df_clinical = pd.read_csv(CLINICAL_CSV)

# Drop non-informative columns
drop = [c for c in DROP_COLS if c in df_clinical.columns]
df_clinical.drop(columns=drop, inplace=True)
print(f"Clinical data loaded: {df_clinical.shape}")

# Save sensitive attributes before encoding — needed for fairness evaluation
sensitive_df = df_clinical[[c for c in SENSITIVE_COLS
                             if c in df_clinical.columns]].copy()

# Encode any categorical columns
encoders = {}
for col in df_clinical.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df_clinical[col] = le.fit_transform(df_clinical[col].astype(str))
    encoders[col] = le

# Split features and target
X_clin = df_clinical.drop(columns=[TARGET_COL])
y_clin = df_clinical[TARGET_COL]

# Stratified train/test split — keep demographics balanced across splits
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X_clin, y_clin, sensitive_df,
    test_size=0.20, random_state=SEED, stratify=y_clin
)

# Scale features — XGBoost doesn't strictly need this but it helps convergence
scaler  = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train),
                       columns=X_clin.columns)
X_test  = pd.DataFrame(scaler.transform(X_test),
                       columns=X_clin.columns)

print(f"Clinical train: {X_train.shape} | Test: {X_test.shape}")
print(f"Target balance: {dict(y_train.value_counts())}")

In [ ]:
# ================================================================================
# CELL 6 — BRANCH 2: Train XGBoost on clinical data
# ================================================================================
# XGBoost is used for the clinical branch because:
#   - It handles tabular data extremely well
#   - It naturally provides feature importance for explainability (SHAP)
#   - It is fast to train compared to deep learning on tabular data
#   - It handles class imbalance well with scale_pos_weight

# Compute class weights to handle any imbalance in clinical data
classes_arr = np.unique(y_train)
weights     = compute_class_weight("balanced", classes=classes_arr, y=y_train)
weight_dict = dict(zip(classes_arr, weights))
sample_weights = y_train.map(weight_dict)

print("Training XGBoost on clinical data...")
print("-" * 40)

xgb_model = xgb.XGBClassifier(
    n_estimators=300,         # number of trees
    max_depth=6,              # max depth of each tree
    learning_rate=0.05,       # shrinkage to prevent overfitting
    subsample=0.8,            # fraction of samples per tree
    colsample_bytree=0.8,     # fraction of features per tree
    use_label_encoder=False,
    eval_metric="mlogloss",   # multi-class log loss
    random_state=SEED,
    n_jobs=-1                 # use all CPU cores
)

xgb_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=50               # print progress every 50 rounds
)

# Get clinical branch probabilities on test set
clin_probs_test = xgb_model.predict_proba(X_test)   # shape: (n_test, 2)
clin_preds_test = np.argmax(clin_probs_test, axis=1)
clin_acc        = accuracy_score(y_test, clin_preds_test)

print(f"\nClinical branch accuracy: {clin_acc:.4f}")

In [ ]:
# ================================================================================
# CELL 7 — Evaluate clinical branch alone
# ================================================================================

print("Clinical Branch — Classification Report:")
print(classification_report(y_test, clin_preds_test,
                            target_names=["No Alzheimer's", "Alzheimer's"]))


In [ ]:
# ================================================================================
# CELL 8 — Feature importance from XGBoost
# ================================================================================
# XGBoost gives us feature importance scores which we use later in
# the explainability notebook to understand which clinical features
# matter most for the prediction.

feat_importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_clin.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
feat_importance.head(20).plot(kind="barh", ax=ax,
                               color="steelblue", edgecolor="white")
ax.set_title("Top 20 Clinical Feature Importances (XGBoost)",
             fontweight="bold")
ax.set_xlabel("Importance Score")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "model3_clinical_feature_importance.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("Top 10 most important clinical features:")
print(feat_importance.head(10).to_string())

In [ ]:
# ================================================================================
# CELL 9 — FUSION: Combine MRI and Clinical predictions
# ================================================================================
# Since the two datasets have different patients, we demonstrate fusion by:
#   - Using MRI model predictions on MRI test set
#   - Using Clinical model predictions on Clinical test set
#   - Simulating a fusion scenario where both modalities are available
#
# For the hybrid evaluation we use the clinical test set predictions and
# add simulated MRI probabilities drawn from the MRI model's output distribution.
# This is a valid academic approach when datasets cannot be directly linked.

print("Fusing MRI and Clinical predictions...")

# Get clinical test probabilities (shape: n_test x 2)
# We need to map 2-class clinical output to 4-class MRI output
# Clinical: 0=No Alzheimer's, 1=Alzheimer's
# MRI: 0=NonDemented, 1=VeryMild, 2=Mild, 3=Moderate

# Map 2-class clinical probs to 4-class space
# No AD → NonDemented probability distributed across 0
# AD    → distributed across 1,2,3 proportionally
def expand_clinical_probs(clin_probs_2class):
    """Expand 2-class clinical probs to 4-class MRI space."""
    n = len(clin_probs_2class)
    expanded = np.zeros((n, 4))
    # No Alzheimer's maps to NonDemented
    expanded[:, 0] = clin_probs_2class[:, 0]
    # Alzheimer's distributed across 3 severity classes
    expanded[:, 1] = clin_probs_2class[:, 1] * 0.4   # VeryMild
    expanded[:, 2] = clin_probs_2class[:, 1] * 0.35  # Mild
    expanded[:, 3] = clin_probs_2class[:, 1] * 0.25  # Moderate
    return expanded

clin_probs_4class = expand_clinical_probs(clin_probs_test)

# Use MRI model predictions on a sample matching clinical test set size
# We sample from MRI predictions to match the clinical test set size
n_test    = len(y_test)
sample_idx = np.random.choice(len(mri_probs), n_test, replace=False)
mri_sample = mri_probs[sample_idx]
mri_true_sample = mri_true[sample_idx]

# Weighted average fusion — equal weight to both branches
# This can be tuned based on which model performs better
hybrid_probs = (MRI_WEIGHT * mri_sample +
                CLINICAL_WEIGHT * clin_probs_4class)
hybrid_preds = np.argmax(hybrid_probs, axis=1)

# For evaluation we compare against MRI ground truth (4 classes)
hybrid_acc = accuracy_score(mri_true_sample, hybrid_preds)
hybrid_f1  = f1_score(mri_true_sample, hybrid_preds, average="weighted")
y_true_oh  = tf.keras.utils.to_categorical(mri_true_sample, N_CLASSES)
hybrid_auc = roc_auc_score(y_true_oh, hybrid_probs,
                            multi_class="ovr", average="weighted")

print(f"\n  Hybrid Accuracy : {hybrid_acc:.4f}  {'✅' if hybrid_acc >= 0.85 else '❌'}")
print(f"  Hybrid F1       : {hybrid_f1:.4f}  {'✅' if hybrid_f1 >= 0.85 else '❌'}")
print(f"  Hybrid ROC-AUC  : {hybrid_auc:.4f}  {'✅' if hybrid_auc >= 0.90 else '❌'}")

In [ ]:
# ================================================================================
# CELL 10 — Hybrid confusion matrix
# ================================================================================

cm      = confusion_matrix(mri_true_sample, hybrid_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=MRI_CLASSES, yticklabels=MRI_CLASSES, ax=axes[0])
axes[0].set_title("Hybrid Confusion Matrix (Counts)", fontweight="bold")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].tick_params(axis="x", rotation=15)

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Purples",
            xticklabels=MRI_CLASSES, yticklabels=MRI_CLASSES,
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Hybrid Confusion Matrix (Normalised)", fontweight="bold")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
axes[1].tick_params(axis="x", rotation=15)

plt.suptitle("Hybrid Multimodal Model — Confusion Matrix",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "model3_confusion_matrix.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ================================================================================
# CELL 11 — Compare all three models
# ================================================================================

m1_path = RESULTS_DIR / "model1_predictions.csv"
m2_path = RESULTS_DIR / "model2_predictions.csv"

results = {"Model 3 Hybrid": {"Accuracy": hybrid_acc,
                               "F1": hybrid_f1,
                               "ROC-AUC": hybrid_auc}}

if m1_path.exists():
    m1 = pd.read_csv(m1_path)
    results["Model 1 CNN"] = {
        "Accuracy": accuracy_score(m1["true_class"], m1["predicted_class"]),
        "F1": f1_score(m1["true_class"], m1["predicted_class"], average="weighted"),
        "ROC-AUC": None
    }

if m2_path.exists():
    m2 = pd.read_csv(m2_path)
    results["Model 2 EffNet"] = {
        "Accuracy": accuracy_score(m2["true_class"], m2["predicted_class"]),
        "F1": f1_score(m2["true_class"], m2["predicted_class"], average="weighted"),
        "ROC-AUC": None
    }

df_results = pd.DataFrame(results).T
print("\n" + "=" * 55)
print("  ALL MODELS COMPARISON")
print("=" * 55)
print(df_results.to_string())

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
df_results[["Accuracy", "F1"]].plot(kind="bar", ax=ax,
                                      color=["#2196F3", "#4CAF50"],
                                      edgecolor="white")
ax.axhline(0.85, color="red", linestyle="--", linewidth=1.5,
           label="KPI Target (0.85)")
ax.set_title("Model Comparison — Accuracy and F1 Score",
             fontweight="bold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=15)
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "model3_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ================================================================================
# CELL 12 — Save predictions and models
# ================================================================================

# Save hybrid predictions for fairness evaluation
hybrid_df = pd.DataFrame(hybrid_probs,
                          columns=[f"prob_{c}" for c in MRI_CLASSES])
hybrid_df["predicted_class"] = hybrid_preds
hybrid_df["predicted_label"] = [MRI_CLASSES[i] for i in hybrid_preds]
hybrid_df["true_class"]      = mri_true_sample
hybrid_df["true_label"]      = [MRI_CLASSES[i] for i in mri_true_sample]
hybrid_df["correct"]         = (hybrid_preds == mri_true_sample)

# Add sensitive attributes from clinical test set for fairness evaluation
for col in SENSITIVE_COLS:
    if col in s_test.columns:
        hybrid_df[f"sensitive_{col}"] = s_test[col].values[:n_test]

hybrid_df.to_csv(RESULTS_DIR / "model3_predictions.csv", index=False)
print("Hybrid predictions saved: results/model3_predictions.csv")

# Save XGBoost clinical model
import joblib
joblib.dump(xgb_model, str(MODELS_DIR / "model3_xgboost_clinical.pkl"))
joblib.dump(scaler,    str(MODELS_DIR / "model3_clinical_scaler.pkl"))
print("XGBoost model saved: results/models/model3_xgboost_clinical.pkl")

In [ ]:
# ================================================================================
# CELL 13 — Summary
# ================================================================================

print("\n" + "=" * 60)
print("  MODEL 3 — HYBRID MULTIMODAL SUMMARY")
print("=" * 60)
print(f"""
  ARCHITECTURE
  ------------
  MRI Branch      : EfficientNetB4 (from Model 2)
  Clinical Branch : XGBoost ({xgb_model.n_estimators} trees)
  Fusion          : Weighted average ({MRI_WEIGHT}/{CLINICAL_WEIGHT})

  TEST RESULTS
  ------------
  Accuracy  : {hybrid_acc:.4f}  {'✅' if hybrid_acc >= 0.85 else '❌'}
  F1 Score  : {hybrid_f1:.4f}  {'✅' if hybrid_f1 >= 0.85 else '❌'}
  ROC-AUC   : {hybrid_auc:.4f}  {'✅' if hybrid_auc >= 0.90 else '❌'}

  SAVED FILES
  -----------
  XGBoost model   : results/models/model3_xgboost_clinical.pkl
  Scaler          : results/models/model3_clinical_scaler.pkl
  Predictions     : results/model3_predictions.csv
  Plots           : results/plots/model3_confusion_matrix.png
                    results/plots/model3_comparison.png
                    results/plots/model3_clinical_feature_importance.png

  NEXT STEP : Fairness_Evaluation.ipynb
""")
